# Notebook 01 — Databricks Ingestion & Delta Lake Write

**Role:** Data Engineer (Databricks side)

**What this does:** A Databricks job that ingests raw transaction data, applies transformations, and writes a Delta table to a shared storage layer (ADLS Gen2 / OneLake).

This notebook runs **directly in Azure Databricks** on a live cluster — it reads the raw CSV and writes the Delta table for real against ADLS Gen2 (no simulation, no Fabric Lakehouse involved on this side).

> 🗣️ **Talking Point:** Databricks is the best-in-class engine for heavy Spark workloads. Data engineers love the open ecosystem, notebook experience, and MLflow. This step stays in Databricks — we do not move it.

In [ ]:
# Storage account backing this demo's ADLS Gen2 containers (bronze/silver) —
# defaults to the account created by deploy.ps1; override via the widget if you redeploy under a different name
dbutils.widgets.text('storage_account_name', 'stbkdemowllakwrlunvf6')
storage_account_name = dbutils.widgets.get('storage_account_name')

bronze_path = f'abfss://bronze@{storage_account_name}.dfs.core.windows.net'
silver_path = f'abfss://silver@{storage_account_name}.dfs.core.windows.net'
print(f'Using storage account: {storage_account_name}')

In [ ]:
# Import PySpark functions and date utilities for the ingestion pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime

# Confirm Spark is available on the Databricks cluster
print(f'Spark version: {spark.version}')
print('Running Databricks ingestion pipeline...')

## Step 1 — Ingest Raw Transactions from Landing Zone

> 🗣️ **Talking Point:** In Databricks, raw files land in ADLS Gen2 or S3. Auto Loader (cloudFiles) picks them up incrementally — exactly-once semantics, schema inference, no manual file management.

In [ ]:
# Read raw CSV transactions directly from the ADLS Gen2 bronze container (the real landing zone Databricks writes from)
raw_txn = spark.read.csv(
    f'{bronze_path}/transactions_raw.csv',
    header=True,
    inferSchema=True
)

# Confirm row count and schema before applying transformations
print(f'Raw records loaded: {raw_txn.count()}')
raw_txn.printSchema()

## Step 2 — Apply Databricks-Style Transformations

> 🗣️ **Talking Point:** Databricks Delta Live Tables (DLT) applies data quality rules declaratively. Here we mirror that pattern manually so you can see what DLT automates.

In [ ]:
# Apply enterprise-grade transformations that mirror what Databricks Delta Live Tables would automate:
# - Parse TransactionDate to a proper date type
# - Add an ingestion timestamp so lineage is traceable
# - Flag large transactions (>= $10,000) for downstream risk review
# - Classify transaction size into amount bands for segmentation
# - Standardise Status to lowercase for consistent filtering
silver_txn = raw_txn \
    .withColumn('TransactionDate', F.to_date('TransactionDate', 'yyyy-MM-dd')) \
    .withColumn('_ingested_at', F.current_timestamp()) \
    .withColumn('_source', F.lit('databricks_ingestion_pipeline')) \
    .withColumn('IsLargeTransaction', F.col('Amount') >= 10000) \
    .withColumn('AmountBand',
        F.when(F.col('Amount') < 500,   'Small')
         .when(F.col('Amount') < 5000,  'Medium')
         .when(F.col('Amount') < 20000, 'Large')
         .otherwise('Very Large')) \
    .withColumn('Status', F.lower(F.col('Status')))

# Write as Delta to the shared ADLS Gen2 'silver' container (external path, not a Databricks-managed table) —
# this is the exact location Fabric reads via its OneLake shortcut, so the hand-off is genuinely zero-copy
silver_table_path = f'{silver_path}/db_silver_transactions'
silver_txn.write.format('delta').mode('overwrite').save(silver_table_path)

# Also register it as an external table in this Databricks workspace for convenient spark.table() access
spark.sql(f"CREATE TABLE IF NOT EXISTS db_silver_transactions USING DELTA LOCATION '{silver_table_path}'")

# Confirm the Delta write succeeded with row count and column list
print(f'Delta table written to: {silver_table_path}')
print(f'Rows: {silver_txn.count()} | Columns: {len(silver_txn.columns)}')
silver_txn.show(5)

## Step 3 — Register in Unity Catalog (Illustrative)

> 🗣️ **Talking Point:** Databricks Unity Catalog is the governance layer — column-level security, lineage, access policies. Microsoft Purview integrates with Unity Catalog so your governance is unified across both platforms. This step prints the registration entry for illustration; wire it up to a real UC metastore if your workspace has one enabled.

In [ ]:
# This cell illustrates what a real Unity Catalog registration would capture — wire this up to
# `ALTER TABLE ... SET TBLPROPERTIES` / UC catalog APIs if your workspace has a Unity Catalog metastore enabled.
# - Column-level access policies (mask PII fields like CustomerID)
# - Lineage tracking (source file -> silver table)
# - Tag-based classification (PII, Financial)
catalog_entry = {
    'catalog':    'banking_prod',
    'schema':     'silver',
    'table':      'transactions',
    'format':     'delta',
    'owner':      'data_engineering_team',
    'tags':       ['financial', 'pii_masked', 'bcbs239'],
    'pii_cols':   ['CustomerID'],
    'masking':    'CustomerID -> SHA256 for non-privileged roles'
}

# Display the Unity Catalog registration entry
print('Unity Catalog Entry:')
for k, v in catalog_entry.items():
    print(f'  {k:12}: {v}')
print()
print('In production: Databricks Unity Catalog syncs lineage to Microsoft Purview automatically.')